# ELEC6604 Assignment 3 — CNN_A, CNN_B and CNN_C (Google Colab)

Run the cells in order. In Colab select **Runtime → Change runtime type → GPU** before training.
This notebook implements **CNN_A** (356,810 parameters) and **CNN_B** (11,644,490 parameters), both without regularization.
Select `MODEL_NAME` in the settings cell; the default is now CNN_C (1,344,106 parameters), with Batch Normalization, Dropout and L2 regularization. No experiment results are pre-filled.

The default experiment runs three seeds for 30 epochs each. For a quick check set `SEEDS = [42]` and `EPOCHS = 1`, then restore the defaults for reported results.
The official test set is used only for final evaluation; validation data comes from the original training set.


In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("No GPU detected. Enable a GPU runtime in Colab for faster training.")


## 1. Experiment settings
Keep the validation split fixed across seeds, so repeats measure training variability.
`NORMALIZATION` can be `zero_one` or `minus_one_one`. To compare normalization later, rerun with the other value while keeping all other settings fixed.
Enable Drive saving to retain outputs after the Colab runtime disconnects. Otherwise download the results ZIP before ending the session.

Set `MODEL_NAME = "CNN_C"` for the regularized model, `"CNN_B"` for the wide/deep model or `"CNN_A"` for the baseline. CNN_C regularization strengths are configurable below. Rerun from the settings cell after changing it. Outputs are saved in separate model-specific directories. For a fair comparison keep split, seeds, normalization and training settings the same.


In [ ]:
MODEL_NAME = "CNN_C"  # "CNN_A", "CNN_B" or "CNN_C"
assert MODEL_NAME in {"CNN_A", "CNN_B", "CNN_C"}
SEEDS = [42, 123, 2026]
SPLIT_SEED = 2026
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
C_L2 = 1e-4
C_BLOCK_DROPOUT = (0.2, 0.3, 0.4)
C_DENSE_DROPOUT = 0.5
NORMALIZATION = "zero_one"
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/ELEC6604_Assignment3")
else:
    OUTPUT_ROOT = Path("/content/assignment3_results")

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = OUTPUT_ROOT / f"{MODEL_NAME}_{NORMALIZATION}_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert NORMALIZATION in {"zero_one", "minus_one_one"}
assert EPOCHS > 0 and len(SEEDS) == len(set(SEEDS)) and len(SEEDS) > 0
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as exc:
    print("Deterministic operations could not be enabled:", exc)
print("Results directory:", OUTPUT_DIR)


## 2. Load and preprocess CIFAR-10
Download directly using Keras; no manually uploaded CIFAR batch files are needed.
Split 50,000 original training images into 45,000 training and 5,000 validation images, stratified by class.
Pixel values are converted to float32 and normalized. Labels are one-hot encoded for categorical cross-entropy.


In [ ]:
(images, labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_indices, val_indices = train_test_split(
    np.arange(len(images)), test_size=0.1, random_state=SPLIT_SEED,
    stratify=labels.ravel(),
)

def normalize(x):
    x = x.astype("float32") / 255.0
    return (x - 0.5) * 2.0 if NORMALIZATION == "minus_one_one" else x

X_train = normalize(images[train_indices])
X_val = normalize(images[val_indices])
X_test = normalize(test_images)
Y_train = keras.utils.to_categorical(labels[train_indices], 10)
Y_val = keras.utils.to_categorical(labels[val_indices], 10)
Y_test = keras.utils.to_categorical(test_labels, 10)
np.savez(OUTPUT_DIR / "split_indices.npz", train=train_indices, validation=val_indices)
del images, test_images
for name, x, y in [("Train", X_train, Y_train), ("Validation", X_val, Y_val), ("Test", X_test, Y_test)]:
    print(name, "images:", x.shape, "labels:", y.shape, "range:", (x.min(), x.max()))
assert (len(X_train), len(X_val), len(X_test)) == (45000, 5000, 10000)
assert not np.intersect1d(train_indices, val_indices).size


## 3. Model architectures
### CNN_A
`32×32×3 → Conv32 → Pool → Conv64 → Pool → Conv128 → Pool → Flatten → Dense128 → Softmax10`

All convolutions use 3×3 kernels, same padding and ReLU; pooling uses 2×2 windows.
There is **no Dropout, Batch Normalization, weight decay, data augmentation or label smoothing**.
Parameter calculation: 896 + 18,496 + 73,856 + 262,272 + 1,290 = **356,810**, within the required 200,000–700,000 range.

### CNN_B
`32×32×3 → Conv64 → Conv64 → Pool → Conv128 → Conv128 → Pool → Conv256 → Conv256 → Pool → Flatten4096 → Dense2048 → Dense1024 → Softmax10`

CNN_B is wider and deeper than CNN_A: six convolutional layers and two large hidden dense layers. All convolutions use 3×3 kernels, same padding and ReLU; pooling uses 2×2 windows. There is no regularization.
Parameters: convolutional layers 1,145,408 + Dense2048 8,390,656 + Dense1024 2,098,176 + classifier 10,250 = **11,644,490**, within the required 10–15 million.
The larger capacity may cause overfitting. Examine the full learning curves instead of assuming it must happen. CNN_B takes longer to train and uses more GPU memory than CNN_A.

### CNN_C
`Input → [Conv32–BN–ReLU]×2 → Pool → Dropout0.2 → [Conv64–BN–ReLU]×2 → Pool → Dropout0.3 → [Conv128–BN–ReLU]×2 → Pool → Dropout0.4 → Flatten2048 → Dense512–BN–ReLU → Dropout0.5 → Softmax10`

CNN_C halves CNN_B's convolutional widths and replaces its 2048/1024 hidden dense layers with one 512-unit layer. L2 kernel regularization (default 0.0001), Batch Normalization and Dropout aim to reduce overfitting; verify this using validation results. No augmentation or label smoothing is used.
Convolutions use 3×3 kernels and same padding; all pooling is 2×2. Layers followed by BN omit bias because BN supplies a learned offset.

Parameter formulas: Conv without bias = kernel height × kernel width × input channels × output channels; Dense without bias = inputs × outputs; BN = 4 × channels (gamma, beta, moving mean, moving variance); output Dense = (inputs + 1) × outputs.

| Layer | Parameters |
|---|---:|
| conv1 | 864 |
| bn1 | 128 |
| conv2 | 9,216 |
| bn2 | 128 |
| conv3 | 18,432 |
| bn3 | 256 |
| conv4 | 36,864 |
| bn4 | 256 |
| conv5 | 73,728 |
| bn5 | 512 |
| conv6 | 147,456 |
| bn6 | 512 |
| dense1 | 1,048,576 |
| bn_dense | 2,048 |
| class_probabilities | 5,130 |

Total **1,344,106** = **1,342,186 trainable** + **1,920 non-trainable** parameters, within 1–5 million. ReLU, pooling, flatten and dropout have no parameters. The next cell verifies each manual count against Keras and exports the table.

**Loss interpretation:** CNN_C loss includes the L2 penalty, so its loss is not directly comparable to A/B's unregularized cross-entropy. Accuracy remains comparable. Dropout/BN behave differently during training and evaluation; use final inference-mode metrics as well as learning curves to discuss the generalization gap.


In [ ]:
def build_cnn_a():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1"),
        layers.MaxPooling2D(2, name="pool1"),
        layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2"),
        layers.MaxPooling2D(2, name="pool2"),
        layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3"),
        layers.MaxPooling2D(2, name="pool3"),
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense1"),
        layers.Dense(10, activation="softmax", name="class_probabilities"),
    ], name="CNN_A")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy", metrics=["accuracy"],
    )
    return model

def build_cnn_b():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(64, 3, padding="same", activation="relu", name="conv1"),
        layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2"),
        layers.MaxPooling2D(2, name="pool1"),
        layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3"),
        layers.Conv2D(128, 3, padding="same", activation="relu", name="conv4"),
        layers.MaxPooling2D(2, name="pool2"),
        layers.Conv2D(256, 3, padding="same", activation="relu", name="conv5"),
        layers.Conv2D(256, 3, padding="same", activation="relu", name="conv6"),
        layers.MaxPooling2D(2, name="pool3"),
        layers.Flatten(name="flatten"),
        layers.Dense(2048, activation="relu", name="dense1"),
        layers.Dense(1024, activation="relu", name="dense2"),
        layers.Dense(10, activation="softmax", name="class_probabilities"),
    ], name="CNN_B")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy", metrics=["accuracy"],
    )
    return model

def build_cnn_c():
    model = keras.Sequential(name="CNN_C")
    model.add(layers.Input(shape=(32, 32, 3)))
    conv_id = 0
    for block, (filters, rate) in enumerate(zip((32, 64, 128), C_BLOCK_DROPOUT), 1):
        for _ in range(2):
            conv_id += 1
            model.add(layers.Conv2D(
                filters, 3, padding="same", use_bias=False,
                kernel_regularizer=keras.regularizers.l2(C_L2), name=f"conv{conv_id}",
            ))
            model.add(layers.BatchNormalization(name=f"bn{conv_id}"))
            model.add(layers.Activation("relu", name=f"relu{conv_id}"))
        model.add(layers.MaxPooling2D(2, name=f"pool{block}"))
        model.add(layers.Dropout(rate, name=f"dropout{block}"))
    model.add(layers.Flatten(name="flatten"))
    model.add(layers.Dense(512, use_bias=False, kernel_regularizer=keras.regularizers.l2(C_L2), name="dense1"))
    model.add(layers.BatchNormalization(name="bn_dense"))
    model.add(layers.Activation("relu", name="relu_dense"))
    model.add(layers.Dropout(C_DENSE_DROPOUT, name="dropout_dense"))
    model.add(layers.Dense(10, activation="softmax", kernel_regularizer=keras.regularizers.l2(C_L2), name="class_probabilities"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy", metrics=["accuracy"],
    )
    return model

MODEL_BUILDERS = {"CNN_A": build_cnn_a, "CNN_B": build_cnn_b, "CNN_C": build_cnn_c}
EXPECTED_PARAMETERS = {"CNN_A": 356810, "CNN_B": 11644490, "CNN_C": 1344106}
PARAMETER_LIMITS = {"CNN_A": (200_000, 700_000), "CNN_B": (10_000_000, 15_000_000), "CNN_C": (1_000_000, 5_000_000)}

def build_selected_model():
    model = MODEL_BUILDERS[MODEL_NAME]()
    lower, upper = PARAMETER_LIMITS[MODEL_NAME]
    assert lower <= model.count_params() <= upper
    assert model.count_params() == EXPECTED_PARAMETERS[MODEL_NAME]
    if MODEL_NAME != "CNN_C":
        assert not model.losses, "CNN_A and CNN_B must not have regularization losses."
    return model

assert len(C_BLOCK_DROPOUT) == 3 and all(0 <= r < 1 for r in C_BLOCK_DROPOUT)
assert 0 <= C_DENSE_DROPOUT < 1 and C_L2 >= 0
preview_model = build_selected_model()
if MODEL_NAME == "CNN_C":
    manual_counts = {'conv1': 864, 'bn1': 128, 'conv2': 9216, 'bn2': 128, 'conv3': 18432, 'bn3': 256, 'conv4': 36864, 'bn4': 256, 'conv5': 73728, 'bn5': 512, 'conv6': 147456, 'bn6': 512, 'dense1': 1048576, 'bn_dense': 2048, 'class_probabilities': 5130}
    parameter_rows = []
    for layer in preview_model.layers:
        expected = manual_counts.get(layer.name, 0)
        actual = layer.count_params()
        assert actual == expected, (layer.name, expected, actual)
        parameter_rows.append({"layer": layer.name, "manual_parameters": expected, "keras_parameters": actual})
    assert sum(manual_counts.values()) == preview_model.count_params()
    parameter_table = pd.DataFrame(parameter_rows)
    parameter_table.to_csv(OUTPUT_DIR / "CNN_C_parameter_calculation.csv", index=False)
    display(parameter_table)

preview_model.summary()
summary_lines = []
preview_model.summary(print_fn=lambda line: summary_lines.append(line))
(OUTPUT_DIR / "model_summary.txt").write_text("\n".join(summary_lines))
try:
    keras.utils.plot_model(preview_model, to_file=str(OUTPUT_DIR / f"{MODEL_NAME}_architecture.png"), show_shapes=True)
except (ImportError, OSError) as exc:
    print("Optional architecture image unavailable:", exc)
del preview_model


## 4. Train and evaluate three seeds
Each run uses a fresh model and optimizer. All runs complete the same number of epochs (no early stopping).
The checkpoint with the lowest validation loss is selected for final evaluation, without consulting test results.
Training time includes validation and checkpoint writing, but excludes data download and final evaluation.
Per-epoch training metrics are measured during updates; the final train/validation metrics below evaluate the selected checkpoint in inference mode.


In [ ]:
config = {
    "model": MODEL_NAME, "seeds": SEEDS, "split_seed": SPLIT_SEED,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "normalization": NORMALIZATION, "tensorflow_version": tf.__version__,
    "devices": [str(d) for d in tf.config.list_physical_devices()],
    "regularization": {"l2": C_L2, "block_dropout": C_BLOCK_DROPOUT, "dense_dropout": C_DENSE_DROPOUT, "batch_normalization": True} if MODEL_NAME == "CNN_C" else None,
    "checkpoint_selection": "minimum validation loss",
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
results, histories = [], {}
for seed in SEEDS:
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    model = build_selected_model()
    run_dir = OUTPUT_DIR / f"seed_{seed}"
    run_dir.mkdir(exist_ok=True)
    checkpoint = run_dir / "best.weights.h5"
    print(f"\nTraining {MODEL_NAME}: seed={seed}")
    start = time.perf_counter()
    history = model.fit(
        X_train, Y_train, validation_data=(X_val, Y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True, verbose=2,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_loss", save_best_only=True, save_weights_only=True),
            keras.callbacks.CSVLogger(str(run_dir / "epochs.csv")),
            keras.callbacks.TerminateOnNaN(),
        ],
    )
    training_seconds = time.perf_counter() - start
    if len(history.history["loss"]) != EPOCHS or not all(
        np.isfinite(values).all() for values in history.history.values()
    ):
        raise RuntimeError("Training failed or produced non-finite metrics; inspect the run before reporting results.")
    histories[seed] = history.history
    (run_dir / "history.json").write_text(json.dumps(history.history, indent=2))
    model.load_weights(checkpoint)
    train_metrics = model.evaluate(X_train, Y_train, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
    val_metrics = model.evaluate(X_val, Y_val, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
    test_metrics = model.evaluate(X_test, Y_test, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
    row = {
        "model": MODEL_NAME, "seed": seed, "parameters": model.count_params(),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_run": len(history.history["loss"]), "training_seconds": training_seconds,
        "seconds_per_epoch": training_seconds / len(history.history["loss"]),
        "train_accuracy": train_metrics["accuracy"], "train_loss": train_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"], "val_loss": val_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"], "test_loss": test_metrics["loss"],
        "train_val_accuracy_gap": train_metrics["accuracy"] - val_metrics["accuracy"],
    }
    results.append(row)
    pd.DataFrame(results).to_csv(OUTPUT_DIR / "results_by_seed.csv", index=False)
    print(row)


## 5. Summarize and plot
Accuracy is stored as a fraction (multiply by 100 for percent). Standard deviation uses the sample definition (`ddof=1`).
With only one smoke-test seed the standard deviation is undefined; these are not the final three-seed results.
Use the curves and train–validation gap to discuss overfitting; do not assume it occurred before examining the results.


In [ ]:
results_df = pd.DataFrame(results)
display(results_df)
metric_columns = ["train_accuracy", "train_loss", "val_accuracy", "val_loss", "test_accuracy", "test_loss", "training_seconds", "seconds_per_epoch", "train_val_accuracy_gap"]
summary_df = results_df[metric_columns].agg(["mean", "std"]).T
summary_df.to_csv(OUTPUT_DIR / "summary_mean_std.csv")
display(summary_df)
if len(SEEDS) != 3:
    print("This is not the required three-seed final experiment.")

fig, axes = plt.subplots(len(SEEDS), 2, figsize=(12, 4 * len(SEEDS)), squeeze=False)
for i, seed in enumerate(SEEDS):
    h = histories[seed]
    epochs = np.arange(1, len(h["loss"]) + 1)
    for j, metric in enumerate(["accuracy", "loss"]):
        ax = axes[i, j]
        ax.plot(epochs, h[metric], label="Training")
        ax.plot(epochs, h["val_" + metric], label="Validation")
        ax.axvline(results[i]["best_epoch"], color="gray", linestyle="--", label="Selected epoch")
        ax.set(title=f"{MODEL_NAME} — seed {seed}", xlabel="Epoch", ylabel=metric.capitalize())
        if metric == "accuracy":
            ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"{MODEL_NAME}_learning_curves.png", dpi=180, bbox_inches="tight")
plt.show()

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Saved experiment ZIP:", archive)


In [ ]:
# Run this cell to download the ZIP to your computer.
from google.colab import files
files.download(archive)


## Report checklist — CNN_A / CNN_B / CNN_C
- State architecture, parameter count, optimizer, learning rate, batch size, normalization and epochs.
- Include the architecture diagram and training/validation curves on the corresponding model report page.
- Report final test accuracy and three-seed mean ± standard deviation, with training time and hardware.
- Explain the checkpoint selection rule and any observed overfitting using actual results.
- Include CNN_C manual parameter calculations and distinguish trainable/non-trainable parameters.
- The normalization comparison and the complete six-page report remain separate work.
